In [2]:
import os
import glob
import numpy as np
import pandas as pd
import librosa
from scipy.signal import welch

# =========================
# CONFIG
# =========================
DATASET_ROOT = "/home/feliciano/Downloads/Preliminary Data/AllFish_2secSplit"
TARGET_SR = 22050
MIN_SAMPLES = 2048   # safety for FFT / Welch

# =========================
# FILE DISCOVERY
# =========================
def get_wav_files(root):
    class_folders = [
        f for f in os.listdir(root)
        if os.path.isdir(os.path.join(root, f))
    ]

    wav_files = {}
    for cls in class_folders:
        files = glob.glob(os.path.join(root, cls, "*.wav"))
        wav_files[cls] = files

    return wav_files

# =========================
# FEATURE EXTRACTION
# =========================
def extract_features(file_path):
    try:
        # Load audio with fixed SR
        y, sr = librosa.load(file_path, sr=TARGET_SR, mono=True)

        if len(y) < MIN_SAMPLES:
            raise ValueError("Audio too short")

        # Duration
        duration = librosa.get_duration(y=y, sr=sr)

        # Time-domain
        zcr = float(np.mean(librosa.feature.zero_crossing_rate(y=y)))
        rms = float(np.mean(librosa.feature.rms(y=y)))

        # FFT (remove DC component)
        fft = np.abs(np.fft.rfft(y))
        freqs = np.fft.rfftfreq(len(y), 1 / sr)
        fft[0] = 0
        fft_mean = float(np.mean(fft))
        fft_peak = float(freqs[np.argmax(fft)])

        # PSD (Welch)
        nperseg = min(1024, len(y))
        f_psd, psd = welch(y, fs=sr, nperseg=nperseg)
        psd_mean = float(np.mean(psd))
        psd_peak = float(f_psd[np.argmax(psd)])

        # Spectral features
        centroid = float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)))
        bandwidth = float(np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr)))
        rolloff = float(np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr)))
        flatness = float(np.mean(librosa.feature.spectral_flatness(y=y)))

        # MFCCs
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        mfcc_means = np.mean(mfccs, axis=1)

        features = {
            "duration": duration,
            "zcr": zcr,
            "rms": rms,
            "fft_mean": fft_mean,
            "fft_peak_freq": fft_peak,
            "psd_mean": psd_mean,
            "psd_peak_freq": psd_peak,
            "centroid": centroid,
            "bandwidth": bandwidth,
            "rolloff": rolloff,
            "flatness": flatness,
        }

        for i, val in enumerate(mfcc_means):
            features[f"mfcc_{i+1}"] = float(val)

        return features

    except Exception as e:
        print(f"Skipping {os.path.basename(file_path)}: {e}")
        return None

# =========================
# DATASET BUILDING
# =========================
def build_dataset():
    wav_files = get_wav_files(DATASET_ROOT)
    rows = []

    for cls, files in wav_files.items():
        for f in files:
            feats = extract_features(f)
            if feats is None:
                continue

            feats["class"] = cls
            feats["file"] = os.path.basename(f)
            rows.append(feats)

    return pd.DataFrame(rows)

# =========================
# RUN PIPELINE
# =========================
df = build_dataset()

# Save raw feature table
df.to_csv("fish_audio_features.csv", index=False)

# Aggregate numeric features only
agg_mean = df.groupby("class").mean(numeric_only=True)
agg_std = df.groupby("class").std(numeric_only=True)

agg_mean.to_csv("fish_features_by_class_mean.csv")
agg_std.to_csv("fish_features_by_class_std.csv")

print("DONE ✅ CSV files saved.")
print("\nMean features (preview):")
print(agg_mean.head())


DONE ✅ CSV files saved.

Mean features (preview):
                      duration       zcr       rms  fft_mean  fft_peak_freq  \
class                                                                         
B1- hydro - 155 fish       2.0  0.036490  0.001844  0.048950      59.446767   
B1-hydro vide              2.0  0.053012  0.001841  0.074418     119.941062   
B2-Hydro - 15 fish         2.0  0.020422  0.005506  0.097649     106.673293   
B2-hydro 10 fish           2.0  0.008945  0.002418  0.023542      59.997238   

                          psd_mean  psd_peak_freq     centroid    bandwidth  \
class                                                                         
B1- hydro - 155 fish  3.318158e-10      61.884661  2354.012087  3237.593933   
B1-hydro vide         2.740941e-10     349.242189   583.959728   409.069545   
B2-Hydro - 15 fish    2.842030e-09     114.156212  1103.645798  1860.329790   
B2-hydro 10 fish      4.604358e-10      64.587090   372.231044   803.213942   

